In [1]:
#!pip install pypdf
#!pip install langchain-community
#!pip install langchain-text-splitters

In [ ]:
#!pip install langchain-chroma
#!pip install langchain-huggingface
#!pip install sentence-transformers

  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
    --------------------------------------- 0.5/23.5 MB 598.6 kB/s eta 0:00:39
   - -------------------------------------- 0.8/23.5 MB 657.6 kB/s eta 0:00:35
   - -------------------------------------- 0.8/23.5 MB 657.6 kB/s eta 0:00:35
   - -------------------------------------- 1.0/23.5 MB 691.4 kB/s eta 0:00:33
   - ----------------------------------

In [ ]:
#from pypdf import PdfReader

#reader = PdfReader(r"C:\Users\Jawad\Documents\MASK AI Internship\Document Semantic Search System\data\raw\AI_Ethics.pdf")

#for page in reader.pages:
#    text = page.extract_text()

In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    r"C:\Users\Jawad\Documents\MASK AI Internship\Document Semantic Search System\data\raw",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print(len(documents))

C:\Users\Jawad\AppData\Local\Temp\ipykernel_18348\3432035121.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


32


In [7]:
documents[23].metadata

{'producer': 'PyPDF2',
 'creator': 'PyPDF',
 'creationdate': '',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'publisher': 'Curran Associates, Inc.',
 'language': 'en-US',
 'created': '2017',
 'eventtype': 'Poster',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-

In [2]:
print(documents[23].page_content)

Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

In [9]:
len(chunks)

428

In [10]:
chunks[0].metadata

{'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV',
 'creator': 'PyPDF',
 'creationdate': '2023-07-11T21:47:41+05:30',
 'moddate': '2023-07-20T16:27:18-04:00',
 'ieee article id': '9844014',
 'ieee issue id': '10190086',
 'subject': 'IEEE Transactions on Artificial Intelligence;2023;4;4;10.1109/TAI.2022.3194503',
 'ieee publication id': '9078688',
 'title': 'An Overview of Artificial Intelligence Ethics',
 'source': 'C:\\Users\\Jawad\\Documents\\MASK AI Internship\\Document Semantic Search System\\data\\raw\\AI_Ethics.pdf',
 'total_pages': 21,
 'page': 0,
 'page_label': '1'}

In [11]:
import os

for i, chunk in enumerate(chunks, start=1):
    chunk.metadata.update({
        "source": os.path.basename(chunk.metadata["source"]),
        "chunk_number": i,
        "document_type": "pdf"
    })

In [16]:
chunks[400].metadata

{'producer': 'PyPDF2',
 'creator': 'PyPDF',
 'creationdate': '',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'publisher': 'Curran Associates, Inc.',
 'language': 'en-US',
 'created': '2017',
 'eventtype': 'Poster',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-

Load the Embedding Model

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

c:\Users\Jawad\Documents\MASK AI Internship\Document Semantic Search System\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Jawad\Documents\MASK AI Internship\Document Semantic Search System\env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jawad\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need t

Create a Persistent Chroma Database

In [19]:
from langchain_chroma import Chroma

vector_db = Chroma(
    collection_name="document_semantic_search",
    embedding_function=embedding_model,
    persist_directory="./chroma_db"
)

every run, insert duplicates.

In [ ]:
#vector_db.add_documents(chunks)

Prevent Duplicate Records

A better approach is to assign a stable ID to every chunk.

In [20]:
ids = []

for chunk in chunks:
    source = chunk.metadata["source"]
    page = chunk.metadata.get("page", 0)
    chunk_num = chunk.metadata["chunk_number"]

    ids.append(f"{source}_{page}_{chunk_num}")

#Add Only New Documents

Check whether each ID already exists before adding it

In [21]:
existing = vector_db.get(ids=ids)

existing_ids = set(existing["ids"])

new_docs = []
new_ids = []

for doc, doc_id in zip(chunks, ids):
    if doc_id not in existing_ids:
        new_docs.append(doc)
        new_ids.append(doc_id)

if new_docs:
    vector_db.add_documents(
        documents=new_docs,
        ids=new_ids
    )
    print(f"Added {len(new_docs)} new chunks.")
else:
    print("Database is already up to date.")

Added 428 new chunks.


Verify Storage

In [22]:
print(vector_db._collection.count())

428


Test Retrieval

In [23]:
results = vector_db.similarity_search(
    "What is Artificial Intelligence?",
    k=3
)

for i, doc in enumerate(results, start=1):
    print("=" * 50)
    print(f"Result {i}")
    print(doc.metadata)
    print(doc.page_content[:300])

Result 1
{'source': 'AI_Ethics.pdf', 'chunk_number': 13, 'document_type': 'pdf', 'total_pages': 21, 'ieee issue id': '10190086', 'creator': 'PyPDF', 'title': 'An Overview of Artificial Intelligence Ethics', 'page': 0, 'ieee article id': '9844014', 'moddate': '2023-07-20T16:27:18-04:00', 'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'subject': 'IEEE Transactions on Artificial Intelligence;2023;4;4;10.1109/TAI.2022.3194503', 'creationdate': '2023-07-11T21:47:41+05:30', 'ieee publication id': '9078688', 'page_label': '1'}
spreading to various disciplines and aspects of our society. AI
is increasingly taking over human tasks and replacing human
decision-making. It has been widely used in a variety of sectors,
such as business, logistics, manufacturing, transportation, health
care, education, state governance, etc.
The
Result 2
{'document_type': 'pdf', 'total_pages': 21, 'chunk_number': 212, 'creationdate': '2023-0

In [26]:
print(doc.page_content)

advancement of AI, current AI systems or agents, such as health-
care robots, have a certain degree of autonomy, intentionality,
and responsibility [22]. Here, the autonomy of AI refers to an AI
system’s ability to operate without human intervention or direct
control. Intentionality refers to the ability that an AI system can
act in a way that is morally harmful or beneﬁcial and the actions
are deliberate and calculated [11]. Responsibility indicates that


In [27]:
results = vector_db.similarity_search_with_score(
    "What is Artificial Intelligence?",
    k=3
)

In [28]:
for i, (doc, score) in enumerate(results, start=1):
    print("=" * 50)
    print(f"Result {i}")
    print(f"Similarity Score: {score}")
    print(doc.metadata)
    print(doc.page_content[:300])

Result 1
Similarity Score: 0.5273239016532898
{'page': 0, 'moddate': '2023-07-20T16:27:18-04:00', 'subject': 'IEEE Transactions on Artificial Intelligence;2023;4;4;10.1109/TAI.2022.3194503', 'ieee article id': '9844014', 'creator': 'PyPDF', 'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'title': 'An Overview of Artificial Intelligence Ethics', 'ieee issue id': '10190086', 'source': 'AI_Ethics.pdf', 'ieee publication id': '9078688', 'creationdate': '2023-07-11T21:47:41+05:30', 'chunk_number': 13, 'page_label': '1', 'document_type': 'pdf', 'total_pages': 21}
spreading to various disciplines and aspects of our society. AI
is increasingly taking over human tasks and replacing human
decision-making. It has been widely used in a variety of sectors,
such as business, logistics, manufacturing, transportation, health
care, education, state governance, etc.
The
Result 2
Similarity Score: 0.5430806875228882
{'producer': '